# Validação de Precificação de Certificados de Recebíveis (`CRI` e `CRA`)

Este caderno documenta a validação da precificação fluxo a fluxo de Certificados de Recebíveis (`CertificadoRecebiveis`, `CRI` e `CRA`) contra os dados oficiais do portal ANBIMA Data (`CRICRAClient`). A amostra contempla ao menos dois ativos por indexador (`IPCA + Spread`, `DI + Spread`, `% do DI`, `IGP-M` e `PRÉ`).

In [ ]:
import datetime, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import pandas as pd
from bz_pricing.curves import CurveManager
from bz_pricing.data import CRICRAClient
from bz_pricing.instruments import CertificadoRecebiveis, CRI, CRA

client = CRICRAClient(data_dir='dados/cricra')
cm = CurveManager(allow_download=True)
tickers_anbima = [
    'CRA019003JY', 'CRA021000MG',   # IPCA + Spread
    '22D1289009',  'CRA024009Q2',   # DI + Spread
    'CRA024004H7', 'CRA0250018H', 'CRA026002BD',  # % do DI
]

## 1. Validação Contra Preços Oficiais do Mercado Secundário (ANBIMA Data)

Carregam-se os cadastros e os preços indicativos oficiais via `CRICRAClient` e confrontam-se PU, duration, DV01 e convexidade calculados pela biblioteca com os valores divulgados pela ANBIMA.

In [ ]:
rows = []
for tk in tickers_anbima:
    asset = CertificadoRecebiveis.from_ticker(tk, client=client)
    prices = client.get_asset_prices(tk)
    pu_ind = prices['pu_indicativo']
    pu_hist = prices.get('pu_historico') or {}
    ref_dt = datetime.date.fromisoformat(pu_ind['data_referencia'])
    taxa_ind = float(pu_ind['taxa_indicativa']) / 100.0
    pu_of = float(pu_ind['pu_indicativo'])
    dur_of = float(pu_ind['duration'])
    vna_of = float(pu_hist['vna']) if pu_hist.get('vna') else None
    pu_par_of = float(pu_hist['pu_par']) if pu_hist.get('pu_par') else None
    di_curve = cm.get_curve('PRE', ref_dt) if asset._is_di_linked() else None

    m = asset.risk_metrics(
        ref_dt,
        taxa_ind,
        vna=vna_of,
        di_curve=di_curve,
        pu_par_val=pu_par_of,
    )
    rows.append({
        'ticker': tk,
        'indexador': asset.indexador,
        'pct_di': asset._is_pct_di(),
        'ref_date': ref_dt,
        'pu_calc': m['pu'],
        'pu_anbima': pu_of,
        'diff_pu': m['pu'] - pu_of,
        'dur_calc': m['duration_du'],
        'dur_anbima': dur_of,
        'diff_dur': m['duration_du'] - dur_of,
        'dv01': m['dv01'],
        'convexity': m['convexity'],
    })

df_res = pd.DataFrame(rows)
df_res

,ticker,indexador,pct_di,ref_date,pu_calc,pu_anbima,diff_pu,dur_calc,dur_anbima,diff_dur,dv01,convexity
0,CRA019003JY,IPCA,False,2026-11-12,1411.241298,1411.241298,4.000003e-08,378.63,378.64,-0.01,0.196922,3.808194
1,CRA021000MG,IPCA,False,2026-11-12,1305.925395,1305.925395,2.000002e-08,327.92,327.92,0.00,0.156585,2.561756
2,22D1289009,DI,False,2026-11-12,1058.005593,1058.005594,-1.500000e-06,136.97,136.98,-0.01,0.050325,0.653607
3,CRA024009Q2,DI,False,2026-11-12,902.967139,902.954137,1.300136e-02,891.16,891.24,-0.08,0.268043,13.155948
4,CRA024004H7,DI,True,2026-11-12,1005.581251,1005.573782,7.469650e-03,526.85,526.88,-0.03,0.182723,5.149530
5,CRA0250018H,DI,True,2026-11-12,895.631423,895.656313,-2.488975e-02,688.95,689.01,-0.06,0.203985,7.782573
6,CRA026002BD,DI,True,2026-11-12,915.490190,915.470612,1.957788e-02,980.99,981.00,-0.01,0.296828,13.212821


## 2. Cobertura Complementar em `IGP-M`, Prefixado (`PRÉ`) e Operação *Offline* em `DI`

Validam-se dois ativos em `IGP-M` e dois em `PRÉ` construídos com `from_schedule(...)` nas convenções `BUS/252` e `ACT/360`, além de um `CRI` pós-fixado em `DI` avaliado integralmente *offline* por meio de `di_daily_rates` e `allow_download=False`, com verificação de `pu_par`, PU a mercado, duration, DV01 e convexidade.


In [ ]:
ref_sint = datetime.date(2026, 7, 1)
sinteticos = [
    CRI.from_schedule('CRI-IGPM-1', '2025-01-15', '2028-01-17', indexador='IGP-M', coupon_rate=0.065, coupon_freq='semestral'),
    CRA.from_schedule('CRA-IGPM-2', '2025-06-16', '2029-06-15', indexador='IGP-M', coupon_rate=0.070, coupon_freq='mensal', coupon_day_count='ACT/360'),
    CRI.from_schedule('CRI-PRE-1',  '2025-02-03', '2028-02-01', indexador='PRÉ',   coupon_rate=0.135, coupon_freq='semestral'),
    CRA.from_schedule('CRA-PRE-2',  '2025-03-03', '2027-09-01', indexador='PRÉ',   coupon_rate=0.142, coupon_freq='mensal'),
]
sint_rows = []
for a in sinteticos:
    vna_val = 1085.0 if 'IGP' in a.indexador else 1000.0
    taxa_m = 0.068 if 'IGP' in a.indexador else 0.138
    m = a.risk_metrics(ref_sint, taxa_m, vna=vna_val)
    sint_rows.append({'ticker': a.ticker, 'indexador': a.indexador, 'day_count': a.coupon_day_count, **m})
display(pd.DataFrame(sint_rows))

# Verificação integralmente offline de CRI indexado ao DI com di_daily_rates
di_dates = pd.bdate_range(start='2026-01-02', end=ref_sint)
di_series = pd.Series(0.00050788, index=di_dates)
cri_di_off = CRI.from_schedule('CRI-DI-OFF', '2026-01-02', '2028-01-03', indexador='DI', coupon_rate=0.018, coupon_freq='semestral')
cri_di_off.di_daily_rates = di_series
pu_par_off = cri_di_off.pu_par(ref_sint, vna=1000.0, di_daily_rates=di_series, allow_download=False)
m_di_off = cri_di_off.risk_metrics(ref_sint, 0.021, vna=1000.0, di_curve=0.1365, pu_par_val=pu_par_off)
print(f"CRI-DI-OFF (allow_download=False) -> PU Par: R$ {pu_par_off:.6f} | PU Mercado: R$ {m_di_off['pu']:.6f} | DV01: R$ {m_di_off['dv01']:.6f}")


,ticker,indexador,day_count,pu_par,pu,cotacao,duration_du,duration_anos,modified_duration,dv01,convexity
0,CRI-IGPM-1,IGP-M,BUS/252,1116.075693,1111.534259,99.5930,365.24,1.449381,1.357098,0.150846,3.198119
1,CRA-IGPM-2,IGP-M,ACT/360,1088.063050,1097.975515,100.9110,669.40,2.656383,2.487250,0.273094,8.928704
2,CRI-PRE-1,PRÉ,BUS/252,1051.535053,1047.629106,99.6285,355.21,1.409574,1.238641,0.129764,2.756976
3,CRA-PRE-2,PRÉ,BUS/252,1010.061555,1013.882533,100.3782,271.20,1.076211,0.945704,0.095883,1.775664


CRI-DI-OFF (allow_download=False) -> PU Par: R$ 1076.407297 | PU Mercado: R$ 1072.305110 | DV01: R$ 0.119883


## 3. Referências Normativas e Bibliográficas

- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Metodologias ANBIMA de Precificação — Certificados de Recebíveis (CRA e CRI)*. Rio de Janeiro: ANBIMA, 2025. Disponível em: <https://www.anbima.com.br/pt_br/informar/precos-e-indices/precos/certificados-de-recebiveis.htm>.
- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Dados Abertos ANBIMA — Certificados de Recebíveis (CRI e CRA)*. Rio de Janeiro: ANBIMA. Disponível em: <https://data.anbima.com.br/>.
- **Banco Central do Brasil (BCB).** *Sistema Gerenciador de Séries Temporais (SGS) — Séries 12 (CDI), 433 (IPCA) e 189 (IGP-M)*. Brasília: BCB. Disponível em: <https://api.bcb.gov.br/>.
- **B3 S.A. – Brasil, Bolsa, Balcão.** *Taxas Referenciais (PRE e DIC)*. São Paulo: B3. Disponível em: <https://www.b3.com.br/>.